In [3]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

from google_play_scraper import app, reviews, Sort 

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [4]:
CBE_APP_ID = "com.combanketh.mobilebanking"
app_info = app(CBE_APP_ID, 
               lang='en', 
               country='et')

print("=" * 50)
print("CBE Bank App Info")
print("=" * 50)
print(f"App Title: {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']}")
print(f"Total Reviews: {app_info['reviews']}")
print(f"Installs: {app_info['installs']}")


CBE Bank App Info
App Title: Commercial Bank of Ethiopia
Current Score: 4.2885904
Total Ratings: 48412
Total Reviews: 9319
Installs: 5,000,000+


# Part 2 Scrapping Reviews

In [5]:
print(f"Scraping reviews for CBE Bank App...")

result, continuation_token = reviews(
    CBE_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST, #Most recent reviews first
    count=500, #Number of reviews to fetch
    filter_score_with=None #Fetch all reviews regardless of rating
)

print(f"Collected {len(result)} raw reviews.")

Scraping reviews for CBE Bank App...
Collected 500 raw reviews.


In [6]:
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"{key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
reviewId: 31cf1f70-1cd8-427c-9cd5-1ccb4113facf
userName: Ademasu Shadaga
userImage: https://play-lh.googleusercontent.com/a/ACg8ocLrd-eKs7L0SOccqFmy2dkRw4f9aUA854jK3dd3wKTIdhoyJQ=mo
content: thanks for you 😘
score: 5
thumbsUpCount: 0
reviewCreatedVersion: None
at: 2026-05-15 20:11:22
replyContent: None
repliedAt: None
appVersion: None


In [7]:
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review': r.get('content', ''),
        'rating': r.get('score', 0),
        'date': r.get('at', ''),
        'bank' : 'Commercial Bank of Ethiopia',
        'souce' : 'Google Play Store'
    })

df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,souce
0,31cf1f70-1cd8-427c-9cd5-1ccb4113facf,thanks for you 😘,5,2026-05-15 20:11:22,Commercial Bank of Ethiopia,Google Play Store
1,7019e213-93dc-4f00-bff3-80cfb80e5d3a,it's okay,4,2026-05-15 19:53:26,Commercial Bank of Ethiopia,Google Play Store
2,ba0c5d66-8085-4bff-908b-f553c7b14ff5,It's not allowing me to transfer money.,2,2026-05-15 12:22:49,Commercial Bank of Ethiopia,Google Play Store
3,377120dd-1d3e-47ad-a715-c0a538c5d33f,IT'S NOT WORK ON HUAWEI DEVICES,4,2026-05-15 12:07:21,Commercial Bank of Ethiopia,Google Play Store
4,46357e27-661d-4136-bf66-ca7bb91e1427,wow,4,2026-05-14 18:52:51,Commercial Bank of Ethiopia,Google Play Store


## Exploring the raw data

In [9]:
#Basic Shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id            object
review               object
rating                int64
date         datetime64[ns]
bank                 object
souce                object
dtype: object


In [10]:
#Rating distribution - what do users think?
print("Rating Distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"{int(rating)} stars: {count:>4} {bar}")  # Scale the bar length

Rating Distribution:
5 stars:  340 ████████████████████████████████████████████████████████████████████
4 stars:   44 ████████
3 stars:   33 ██████
2 stars:   12 ██
1 stars:   71 ██████████████


In [11]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-15 20:11:22
1   2026-05-15 19:53:26
2   2026-05-15 12:22:49
3   2026-05-15 12:07:21
4   2026-05-14 18:52:51
5   2026-05-14 18:28:23
6   2026-05-14 16:49:46
7   2026-05-14 16:46:29
8   2026-05-14 12:13:13
9   2026-05-14 10:53:56

Date dtype: datetime64[ns]


## Data Quality Audit

In [16]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("="*50)

#___ Problem 1: Missing Values ___
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"{col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
review_id      : OK
review         : OK
rating         : OK
date           : OK
bank           : OK
souce          : OK


In [18]:
# Duplicate reviews
print("\nProblem 2: Duplicate")
print("-" * 30)

#Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review'], keep=False)
print(f"Exact duplicate reviews: {exact_dupes.sum()}")

#Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id'], keep=False)
print(f"Duplicate review IDs: {id_dupes.sum()}")

#Empty reviews
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"Empty reviews: {empty_reviews}")


Problem 2: Duplicate
------------------------------
Exact duplicate reviews: 148
Duplicate review IDs: 0
Empty reviews: 0
